<a href="https://colab.research.google.com/github/korzhimanov/dsp-seminars/blob/main/seminars/5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Практическое занятие №5
## Основные характеристики цифровых фильтров

## Импорт библиотек

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from scipy.signal import savgol_filter, savgol_coeffs, group_delay
import ipywidgets as widgets
from IPython.display import display

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 5)

# Вспомогательная функция для построения частотных характеристик
def plot_filter_response(b, a, fs=1000, title=''):
    """Строит АЧХ (линейный и дБ), ФЧХ и групповую задержку фильтра"""
    w, H = signal.freqz(b, a, fs=fs)
    mag_dB = 20*np.log10(np.abs(H) + 1e-12)
    phase = np.unwrap(np.angle(H))
    w_gd, gd = signal.group_delay((b, a), fs=fs)

    plt.figure(figsize=(12,10))
    plt.subplot(4,1,1)
    plt.semilogy(w, np.abs(H))
    plt.title(f'АЧХ (линейный масштаб) – {title}')
    plt.grid()
    plt.subplot(4,1,2)
    plt.plot(w, mag_dB)
    plt.title('АЧХ (дБ)')
    plt.ylabel('|H| (дБ)')
    plt.grid()
    plt.subplot(4,1,3)
    plt.plot(w, phase)
    plt.title('ФЧХ')
    plt.ylabel('Фаза (рад)')
    plt.grid()
    plt.subplot(4,1,4)
    plt.plot(w_gd, gd)
    plt.title('Групповая задержка')
    plt.xlabel('Частота (Гц)')
    plt.ylabel('Задержка (отсчёты)')
    plt.grid()
    plt.tight_layout()
    plt.show()


## Часть 1. КИХ-фильтр: скользящее среднее

**Скользящее среднее** – простейший КИХ-фильтр. Его коэффициенты: `b_k = 1/M` для `k = 0,...,M-1`. Фильтр имеет линейную фазу.

### Задание 1.1. Импульсная и переходная характеристики

Для длины окна `M = 5`:
1. Сформируйте массив коэффициентов `b` (нормированный).
2. Создайте тестовые сигналы: единичный импульс (длиной 20 отсчётов) и единичный скачок (длиной 50 отсчётов).
3. Примените фильтр с помощью `signal.lfilter(b, a, x)`, где `a = [1]`.
4. Постройте импульсную и переходную характеристики.


In [ ]:
# Параметры скользящего среднего
M = 5
b = np.full(M, 1 / M)
a = np.array([1.0])

# Импульсная характеристика
x_imp = np.zeros(20)
x_imp[0] = 1
h = signal.lfilter(b, a, x_imp)

# Переходная характеристика
x_step = np.ones(50)
s = signal.lfilter(b, a, x_step)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].stem(np.arange(len(h)), h, basefmt=' ')
ax[0].set_title(f'Импульсная характеристика, M={M}')
ax[0].set_xlabel('n')
ax[0].set_ylabel('h[n]')
ax[0].grid(True)

ax[1].plot(s, lw=2)
ax[1].set_title('Переходная характеристика')
ax[1].set_xlabel('n')
ax[1].set_ylabel('s[n]')
ax[1].grid(True)

plt.tight_layout()
plt.show()


**Вопрос:** Почему импульсная характеристика имеет конечную длину, а переходная выходит на постоянный уровень?


**Ответ:**
У КИХ-фильтра отклик на единичный импульс всегда конечный: он равен набору коэффициентов фильтра и после длины окна обнуляется.

Переходная характеристика выходит на плато, потому что при постоянном входе фильтр усредняет одинаковые значения. После заполнения окна выход перестаёт меняться.


### Задание 1.2. АЧХ и ФЧХ

Используя функцию `plot_filter_response`, постройте характеристики для скользящего среднего с длинами окон `M = 5`, `M = 10`, `M = 20`.


In [ ]:
for M in [5, 10, 20]:
    b = np.full(M, 1 / M)
    plot_filter_response(b, [1], fs=1000, title=f'Скользящее среднее, M={M}')


**Вопрос:** Как изменение `M` влияет на крутизну среза АЧХ и на ФЧХ?


**Ответ:**
При увеличении длины окна `M` фильтр становится более "инерционным": высокие частоты подавляются сильнее, а переходная область АЧХ сдвигается к меньшим частотам.

ФЧХ остаётся линейной (это линейно-фазный КИХ), но наклон увеличивается, значит растёт групповая задержка: примерно `(M-1)/2` отсчёта.


### Задание 1.3. Фильтрация суммы синусоид

Сгенерируйте сигнал длительностью 1 с (fs=1000 Гц): `x = sin(2π·10·t) + 0.5·sin(2π·100·t)`. Примените скользящее среднее с `M=5` и `M=20`. Постройте временные графики и спектры (амплитудный спектр через БПФ).


In [ ]:
fs = 1000
t = np.arange(0, 1, 1/fs)
x = np.sin(2*np.pi*10*t) + 0.5*np.sin(2*np.pi*100*t)

# Фильтрация
y_M5 = signal.lfilter(np.ones(5)/5, [1], x)
y_M20 = signal.lfilter(np.ones(20)/20, [1], x)

# Временные графики
plt.figure(figsize=(12, 4))
plt.plot(t[:120], x[:120], label='Исходный', alpha=0.8)
plt.plot(t[:120], y_M5[:120], label='После M=5')
plt.plot(t[:120], y_M20[:120], label='После M=20')
plt.title('Сигнал во времени (первые отсчёты)')
plt.xlabel('Время, с')
plt.ylabel('Амплитуда')
plt.grid(True)
plt.legend()
plt.show()

# Спектры

def amp_spectrum(sig):
    X = np.fft.rfft(sig)
    f = np.fft.rfftfreq(len(sig), d=1/fs)
    A = 2*np.abs(X)/len(sig)
    return f, A

plt.figure(figsize=(12, 4))
for s, lbl in [(x, 'Исходный'), (y_M5, 'M=5'), (y_M20, 'M=20')]:
    f, A = amp_spectrum(s)
    plt.plot(f, A, label=lbl)

plt.xlim(0, 150)
plt.title('Амплитудные спектры')
plt.xlabel('Частота, Гц')
plt.ylabel('Амплитуда')
plt.grid(True)
plt.legend()
plt.show()


**Вопрос:** Как значение `M` влияет на амплитуду 100 Гц сигнала? Объясните, почему так. Какая связь между АЧХ и результатом фильтрации?


**Ответ:**
Компонента 100 Гц уменьшается тем сильнее, чем больше `M`: для `M=20` она подавляется заметно сильнее, чем для `M=5`.

Причина прямая: фильтрация каждой гармоники равна умножению её амплитуды на `|H(f)|`. У скользящего среднего `|H(f)|` быстрее падает на высоких частотах при увеличении окна.


## Часть 2. БИХ-фильтр: экспоненциальное сглаживание

Фильтр первого порядка: `y[n] = α x[n] + (1-α) y[n-1]`. Коэффициенты: `b = [α]`, `a = [1, -(1-α)]`.

### Задание 2.1. Реализация и характеристики

Для `α = 0.2`:
1. Постройте импульсную характеристику (50 отсчётов) и переходную характеристику.
2. Используя `plot_filter_response`, постройте АЧХ, ФЧХ и групповую задержку.


In [ ]:
alpha = 0.2
b = [alpha]
a = [1, -(1-alpha)]

# Импульсная характеристика (длина 50)
impulse = np.zeros(50)
impulse[0] = 1
h = signal.lfilter(b, a, impulse)
# Переходная характеристика (длина 100)
step = np.ones(100)
s = signal.lfilter(b, a, step)

plt.figure(figsize=(12,4))
plt.subplot(1,2,1)
plt.stem(h, basefmt=" ")
plt.title(f'Импульсная характеристика (α={alpha})')
plt.grid()
plt.subplot(1,2,2)
plt.plot(s)
plt.title('Переходная характеристика')
plt.grid()
plt.show()

# Частотные характеристики
plot_filter_response(b, a, fs=1000, title=f'Экспоненциальный α={alpha}')


### Задание 2.2. Влияние параметра α

Исследуйте фильтр при `α = 0.05, 0.2, 0.8`. Постройте на одном графике АЧХ (в дБ) и на другом – групповую задержку. Сделайте выводы.


In [ ]:
alphas = [0.05, 0.2, 0.8]

fig, ax = plt.subplots(2, 1, figsize=(12, 8))

for alpha in alphas:
    b = [alpha]
    a = [1, -(1 - alpha)]

    f, H = signal.freqz(b, a, fs=1000)
    f_gd, gd = group_delay((b, a), fs=1000)

    ax[0].plot(f, 20*np.log10(np.abs(H) + 1e-12), label=f'α={alpha}')
    ax[1].plot(f_gd, gd, label=f'α={alpha}')

ax[0].set_title('АЧХ экспоненциального фильтра (дБ)')
ax[0].set_ylabel('|H|, дБ')
ax[0].grid(True)
ax[0].legend()

ax[1].set_title('Групповая задержка')
ax[1].set_xlabel('Частота, Гц')
ax[1].set_ylabel('Задержка, отсчёты')
ax[1].grid(True)
ax[1].legend()

plt.tight_layout()
plt.show()


**Вопрос:** Как α влияет на скорость реакции фильтра на изменение сигнала и на крутизну спада АЧХ?


**Ответ:**
Чем меньше `α`, тем сильнее сглаживание и тем медленнее фильтр реагирует на изменения сигнала.

Для малого `α` частота среза ниже, подавление высоких частот сильнее, но задержка больше. Для большого `α` всё наоборот: фильтр быстрее, но сглаживание слабее.


### Задание 2.3. Фильтрация зашумлённого сигнала

Сгенерируйте сигнал: синусоида 20 Гц + белый шум (SNR ≈ 5 дБ), fs=500 Гц. Примените экспоненциальный фильтр с α = 0.1 и α = 0.5. Постройте исходный и отфильтрованные сигналы (первые 0.2 с). Сравните степень сглаживания и задержку.


In [ ]:
fs = 500
t = np.arange(0, 1, 1/fs)
x_clean = np.sin(2*np.pi*20*t)

# Шум с SNR примерно 5 дБ
P_sig = np.mean(x_clean**2)
snr_db = 5
P_noise = P_sig / (10**(snr_db/10))
noise = np.sqrt(P_noise) * np.random.randn(len(t))
x_noisy = x_clean + noise

# Фильтрация
y_a01 = signal.lfilter([0.1], [1, -0.9], x_noisy)
y_a05 = signal.lfilter([0.5], [1, -0.5], x_noisy)

plt.figure(figsize=(12, 4))
plt.plot(t[:120], x_noisy[:120], color='gray', alpha=0.6, label='Зашумлённый')
plt.plot(t[:120], x_clean[:120], 'k--', label='Чистый')
plt.plot(t[:120], y_a01[:120], label='α=0.1')
plt.plot(t[:120], y_a05[:120], label='α=0.5')
plt.title('Экспоненциальное сглаживание')
plt.xlabel('Время, с')
plt.ylabel('Амплитуда')
plt.grid(True)
plt.legend()
plt.show()


**Вопрос:** Как на временных графиках проявляется групповая задержка, вносимая БИХ-фильтром?


**Ответ:**
Групповая задержка видна как сдвиг отфильтрованного сигнала вправо относительно исходного.

На графике это заметно по положению максимумов: при `α=0.1` запаздывание больше, чем при `α=0.5`, потому что фильтр сильнее усредняет прошлые отсчёты.


## Часть 3. Фильтр Савицкого – Голея (Savitzky–Golay)

Этот КИХ-фильтр основан на локальной полиномиальной регрессии. Он сохраняет форму сигнала (пики, перегибы) лучше, чем скользящее среднее.

### Задание 3.1. Базовое применение и влияние параметров

Сгенерируйте синусоиду 5 Гц (fs=100 Гц, длительность 2 с) и добавьте белый шум (дисперсия 0.2). Примените `savgol_filter` с параметрами:
- (window=11, polyorder=2)
- (window=21, polyorder=2)
- (window=11, polyorder=5)

Постройте исходный чистый сигнал, зашумлённый и три сглаженные версии. Объясните влияние параметров.


In [ ]:
fs = 100
t = np.arange(0, 2, 1/fs)
x_clean = np.sin(2*np.pi*5*t)
x_noisy = x_clean + 0.2*np.random.randn(len(t))

# Разные комбинации параметров
y_11_2 = savgol_filter(x_noisy, window_length=11, polyorder=2)
y_21_2 = savgol_filter(x_noisy, window_length=21, polyorder=2)
y_11_4 = savgol_filter(x_noisy, window_length=11, polyorder=4)

plt.figure(figsize=(12, 6))
plt.plot(t, x_noisy, color='gray', alpha=0.35, label='Зашумлённый')
plt.plot(t, x_clean, 'k--', lw=2, label='Исходный')
plt.plot(t, y_11_2, label='win=11, poly=2')
plt.plot(t, y_21_2, label='win=21, poly=2')
plt.plot(t, y_11_4, label='win=11, poly=4')
plt.title('Savitzky–Golay: влияние параметров')
plt.xlabel('Время, с')
plt.ylabel('Амплитуда')
plt.grid(True)
plt.legend()
plt.show()


**Вопрос 3.1:** Как увеличение `window_length` и увеличение `polyorder` влияют на степень сглаживания и сохранение формы сигнала?


**Ответ:**
Рост `window_length` усиливает сглаживание: случайные колебания подавляются лучше, но мелкие детали могут "подъедаться".

Увеличение `polyorder` при том же окне обычно лучше сохраняет форму (кривизну) сигнала, но может немного снизить степень подавления шума.


### Задание 3.2. Сравнение со скользящим средним

Создайте сигнал с острым пиком (гауссов импульс с σ=3) на фоне шума. Примените скользящее среднее (window=11) и Savitzky–Golay (window=11, polyorder=2). Сравните, какой фильтр лучше сохраняет форму пика.


In [ ]:
# Сигнал с острым пиком
N = 100
n = np.arange(N)
sigma = 3
x_clean_pulse = np.exp(-((n - N//2)**2) / (2*sigma**2))
x_clean_pulse /= x_clean_pulse.max()
x_noisy_pulse = x_clean_pulse + 0.1*np.random.randn(N)

# Два фильтра
h_ma = np.ones(11) / 11
y_ma = np.convolve(x_noisy_pulse, h_ma, mode='same')
y_sg = savgol_filter(x_noisy_pulse, window_length=11, polyorder=2)

plt.figure(figsize=(12, 5))
plt.plot(x_noisy_pulse, color='gray', alpha=0.5, label='Зашумлённый')
plt.plot(x_clean_pulse, 'k--', lw=2, label='Исходный пик')
plt.plot(y_ma, label='Скользящее среднее (11)')
plt.plot(y_sg, label='Savitzky–Golay (11,2)')
plt.title('Сохранение пика после фильтрации')
plt.xlabel('Отсчёт')
plt.ylabel('Амплитуда')
plt.grid(True)
plt.legend()
plt.show()

# Небольшая численная оценка
peak_idx = np.argmax(x_clean_pulse)
print('Высота пика:')
print(f'  исходный: {x_clean_pulse[peak_idx]:.3f}')
print(f'  MA:       {y_ma[peak_idx]:.3f}')
print(f'  SG:       {y_sg[peak_idx]:.3f}')


**Вопрос:** Какой из фильтров лучше сохраняет пик?


**Ответ:**
В этой постановке лучше сохраняет пик фильтр Savitzky–Golay: вершина и форма импульса ближе к исходной.

Скользящее среднее сильнее размывает локальный максимум, потому что выполняет простое усреднение в окне.


### Задание 3.3. Частотные характеристики

Получите коэффициенты Savitzky–Golay для `window_length=11, polyorder=2` с помощью `savgol_coeffs`. Нормируйте их так, чтобы сумма была равна 1. Постройте АЧХ этого фильтра и сравните с АЧХ скользящего среднего той же длины.


In [ ]:
window_length = 11
polyorder = 2
h_sg = savgol_coeffs(window_length, polyorder)
h_sg = h_sg / np.sum(h_sg)

h_ma = np.ones(window_length) / window_length

f_sg, H_sg = signal.freqz(h_sg, [1], fs=1000)
f_ma, H_ma = signal.freqz(h_ma, [1], fs=1000)

plt.figure(figsize=(12, 5))
plt.plot(f_sg, 20*np.log10(np.abs(H_sg)+1e-12), label='Savitzky–Golay (11,2)')
plt.plot(f_ma, 20*np.log10(np.abs(H_ma)+1e-12), label='Скользящее среднее (11)')
plt.title('Сравнение АЧХ фильтров')
plt.xlabel('Частота, Гц')
plt.ylabel('|H(f)|, дБ')
plt.xlim(0, 250)
plt.grid(True)
plt.legend()
plt.show()


**Вопрос:** Чем отличается АЧХ Savitzky–Golay от АЧХ скользящего среднего? Как это объясняет сохранение формы сигнала?


**Ответ:**
У Savitzky–Golay АЧХ в полосе малых частот обычно более "щадящая", поэтому форма медленно меняющегося сигнала сохраняется лучше.

Скользящее среднее сильнее срезает высокочастотные составляющие, но вместе с шумом сильнее сглаживает и полезные детали.


### Задание 3.4. Фильтрация меандра

Сгенерируйте меандр частотой 2 Гц (fs=500 Гц) и добавьте синусоидальную помеху 50 Гц амплитудой 0.5. Примените Savitzky–Golay (window=15, polyorder=3) и скользящее среднее (window=15). Сравните, как подавляется помеха и сохраняются фронты.


In [ ]:
fs = 500
t = np.arange(0, 1, 1/fs)
meander = signal.square(2*np.pi*2*t)
interference = 0.5 * np.sin(2*np.pi*50*t)
x_meander = meander + interference

# Фильтрация
h_ma15 = np.ones(15) / 15
y_ma = np.convolve(x_meander, h_ma15, mode='same')
y_sg = savgol_filter(x_meander, window_length=15, polyorder=3)

plt.figure(figsize=(12, 5))
plt.plot(t[:220], x_meander[:220], label='Исходный (меандр + 50 Гц)')
plt.plot(t[:220], y_ma[:220], label='Скользящее среднее (15)')
plt.plot(t[:220], y_sg[:220], label='Savitzky–Golay (15,3)')
plt.title('Фильтрация меандра: сравнение фронтов')
plt.xlabel('Время, с')
plt.ylabel('Амплитуда')
plt.grid(True)
plt.legend()
plt.show()


**Вопрос:** Какой фильтр лучше подходит для обработки сигналов с резкими фронтами? Почему?


**Ответ:**
Для сигналов с резкими фронтами обычно предпочтительнее Savitzky–Golay: он лучше удерживает форму переходов и меньше "скругляет" ступеньки.

Скользящее среднее тоже убирает помеху, но фронты после него становятся более пологими.


## Часть 4. Сравнение КИХ и БИХ фильтров

Для скользящего среднего (`M=10`), экспоненциального фильтра (`α=0.2`) и Savitzky–Golay (`win=11, polyorder=2`):

1. Постройте на одном графике их АЧХ (в дБ) и групповую задержку.
2. Подайте на них единичный импульс и единичный скачок, постройте реакции.
3. Сгенерируйте сигнал из прямоугольных импульсов с шумом (как в примере ниже) и примените все три фильтра. Оцените качество подавления шума и сохранения фронтов.


In [ ]:
# Сигнал с прямоугольными импульсами
fs = 1000
t = np.arange(0, 1, 1/fs)
pulse_train = np.zeros_like(t)
pulse_train[0:100] = 1
pulse_train[250:350] = 1
pulse_train[500:600] = 1
pulse_train[750:850] = 1
x_noisy = pulse_train + 0.1*np.random.randn(len(t))

# Параметры фильтров
h_ma = np.ones(10) / 10
b_exp = [0.2]
a_exp = [1, -0.8]
h_sg = savgol_coeffs(11, 2)
h_sg = h_sg / np.sum(h_sg)

# 1) АЧХ и групповая задержка
fig, ax = plt.subplots(2, 1, figsize=(12, 8))
for b, a, name in [
    (h_ma, [1], 'MA (M=10)'),
    (b_exp, a_exp, 'Экспоненциальный (α=0.2)'),
    (h_sg, [1], 'Savitzky–Golay (11,2)')
]:
    f, H = signal.freqz(b, a, fs=fs)
    fg, gd = group_delay((b, a), fs=fs)
    ax[0].plot(f, 20*np.log10(np.abs(H)+1e-12), label=name)
    ax[1].plot(fg, gd, label=name)

ax[0].set_title('АЧХ (дБ)')
ax[0].set_ylabel('|H|, дБ')
ax[0].grid(True)
ax[0].legend()

ax[1].set_title('Групповая задержка')
ax[1].set_xlabel('Частота, Гц')
ax[1].set_ylabel('Отсчёты')
ax[1].grid(True)
ax[1].legend()
plt.tight_layout()
plt.show()

# 2) Импульсная и переходная характеристики
imp = np.zeros(60); imp[0] = 1
step = np.ones(60)

resp = {
    'MA (M=10)': (signal.lfilter(h_ma, [1], imp), signal.lfilter(h_ma, [1], step)),
    'Эксп. (α=0.2)': (signal.lfilter(b_exp, a_exp, imp), signal.lfilter(b_exp, a_exp, step)),
    'SG (11,2)': (signal.lfilter(h_sg, [1], imp), signal.lfilter(h_sg, [1], step)),
}

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for name, (h, s) in resp.items():
    ax[0].plot(h, label=name)
    ax[1].plot(s, label=name)

ax[0].set_title('Импульсные отклики')
ax[0].set_xlabel('n')
ax[0].set_ylabel('h[n]')
ax[0].grid(True)
ax[0].legend()

ax[1].set_title('Переходные отклики')
ax[1].set_xlabel('n')
ax[1].set_ylabel('s[n]')
ax[1].grid(True)
ax[1].legend()

plt.tight_layout()
plt.show()

# 3) Фильтрация зашумлённого сигнала
y_ma = signal.lfilter(h_ma, [1], x_noisy)
y_exp = signal.lfilter(b_exp, a_exp, x_noisy)
y_sg = savgol_filter(x_noisy, window_length=11, polyorder=2)

plt.figure(figsize=(12, 5))
plt.plot(t, x_noisy, color='lightgray', label='Зашумлённый сигнал')
plt.plot(t, pulse_train, 'k--', lw=1.5, label='Идеальные импульсы')
plt.plot(t, y_ma, label='MA (M=10)')
plt.plot(t, y_exp, label='Экспоненциальный (α=0.2)')
plt.plot(t, y_sg, label='Savitzky–Golay (11,2)')
plt.title('Сравнение фильтров на импульсном сигнале')
plt.xlabel('Время, с')
plt.ylabel('Амплитуда')
plt.grid(True)
plt.legend()
plt.show()


**Вопрос:** Какой фильтр обеспечивает наилучшее подавление шума? Какой лучше сохраняет фронты? Какой имеет наименьшие фазовые искажения?


**Ответ:**
По этому примеру:

1. Наиболее сильное подавление случайного шума показывает скользящее среднее (и близко к нему экспоненциальный фильтр при малом `α`).
2. Лучшее сохранение фронтов и формы импульсов показывает Savitzky–Golay.
3. По фазовым искажениям преимущество у линейно-фазных КИХ (скользящее среднее и SG как симметричный КИХ). У БИХ-фильтра фаза более нелинейная.

Итог: для "жёсткого" шумоподавления лучше агрессивное сглаживание, для сохранения геометрии сигнала — Savitzky–Golay.
